In [1]:
import pandas as pd
import numpy as np

In [2]:
fred = pd.read_csv('fred_unemployment_raw.csv')
elections = pd.read_csv('election_results_1976_2024.csv')

In [3]:
fred.head()

,date,state,series_id,unemployment_rate,realtime_start,realtime_end
0,1976-01-01,Alabama,ALUR,6.7,2026-05-05,2026-05-05
1,1976-02-01,Alabama,ALUR,6.6,2026-05-05,2026-05-05
2,1976-03-01,Alabama,ALUR,6.600,2026-05-05,2026-05-05
3,1976-04-01,Alabama,ALUR,6.500,2026-05-05,2026-05-05
4,1976-05-01,Alabama,ALUR,6.400,2026-05-05,2026-05-05


In [4]:
elections.head()

,Year,State,Region,Democratic %,Republican %,P.S.,P.S. Score,National Winner
0,1976,Alabama,S,55.7,42.6,D+,11.23,Carter
1,1976,Alaska,W,35.7,57.9,R+,-25.89,Carter
2,1976,Arizona,W,39.8,56.4,R+,-19.33,Carter
3,1976,Arkansas,S,64.9,34.9,D+,27.94,Carter
4,1976,California,W,47.6,49.3,R+,-3.94,Carter


In [ ]:
elections["State"] = elections["State"].str.strip().str.title()
fred["state"] = fred["state"].str.strip().str.title()

In [ ]:
elections = elections[elections["State"] != "Washington D.C."]

In [ ]:
fred["date"] = pd.to_datetime(fred["date"])
fred["Year"] = fred["date"].dt.year
fred["Month"] = fred["date"].dt.month

In [ ]:
fred_nov = fred[fred["Month"] == 11].copy()

In [ ]:
fred_nov = fred_nov[["Year", "state", "unemployment_rate"]]
fred_nov = fred_nov.rename(columns={"state": "State"})

In [ ]:
fused = pd.merge(
    elections,
    fred_nov,
    on=["Year", "State"],
    how="left"
)

In [ ]:
fused["unemployment_rate"] = pd.to_numeric(fused["unemployment_rate"], errors="coerce")

In [ ]:
fused = fused.sort_values(["State", "Year"])

fused["unemployment_change"] = (
    fused.groupby("State")["unemployment_rate"].diff()
)

In [ ]:
fused = fused.rename(columns={
    "unemployment_rate": "ElectionYear_Unemployment",
    "unemployment_change": "Change_Since_Last_Election"
})

In [ ]:
fused.head()

In [ ]:
fused.to_csv("fused.csv")

In [ ]:
import hashlib
with open("fused.csv", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
with open("fused.csv.sha", 'w') as f:
    f.write(sha256)